# Converting a trained model to tflite
https://www.tensorflow.org/lite/microcontrollers/build_convert#model_conversion

# Convert model to tflite

In [1]:
import tensorflow as tf
import numpy as np

I0000 00:00:1774553353.901234   16404 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774553354.661603   16404 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774553357.705928   16404 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
training_spectrogram = np.load('training_spectrogram.npz')
validation_spectrogram = np.load('validation_spectrogram.npz')
test_spectrogram = np.load('test_spectrogram.npz')

X_train = training_spectrogram['X']
X_validate = validation_spectrogram['X']
X_test = test_spectrogram['X']

complete_train_X = np.concatenate((X_train, X_validate, X_test))
# complete_train_X = X_validate

In [4]:
model = tf.keras.models.load_model("fully_trained.model.keras")
converter2 = tf.lite.TFLiteConverter.from_keras_model(model)
converter2.optimizations = [tf.lite.Optimize.DEFAULT]
def representative_dataset_gen():
    for i in range(0, len(complete_train_X), 100):
        # Get sample input data as a numpy array in a method of your choosing.
        yield [complete_train_X[i:i+100]]
converter2.representative_dataset = representative_dataset_gen
# converter.optimizations = [tf.lite.Optimize.OPTIMIZE_FOR_SIZE]
converter2.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
tflite_quant_model = converter2.convert()
open("converted_model.tflite", "wb").write(tflite_quant_model)

E0000 00:00:1774553564.720373   16404 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


INFO:tensorflow:Assets written to: /tmp/tmpfy8kqf29/assets


INFO:tensorflow:Assets written to: /tmp/tmpfy8kqf29/assets


Saved artifact at '/tmp/tmpfy8kqf29'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 99, 43, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  128171004141328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128171004143824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128171004144208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128171004143248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128171004137680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128171004137872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128171004138064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128171004145168: TensorSpec(shape=(), dtype=tf.resource, name=None)


/home/gerard/ai-voice-control/model/.venv/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1774553565.659711   16404 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1774553565.659726   16404 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1774553565.660808   16404 reader.cc:83] Reading SavedModel from: /tmp/tmpfy8kqf29
I0000 00:00:1774553565.661340   16404 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1774553565.661347   16404 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpfy8kqf29
I0000 00:00:1774553565.666742   16404 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1774553565.667657   16404 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1774553565.717499   16404 loader.cc:220] Running initializati

85328

# To convert to C++
This will run a command line too to convert out tflite model into C code.

In [5]:
!xxd -i converted_model.tflite > model_data.cc